In [ ]:
!pip install -q langchain langchain-text-splitters langchain-chroma langchain-huggingface pypdf sentence-transformers
!pip install -q transformers accelerate bitsandbytes

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 855.7 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 49.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 64.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 51.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.9/178.9 kB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.9/61.9 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/

In [ ]:
!pip install -q langchain-community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 36.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
google-adk 2.3.0 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.43.0 which is incompatible.
google-adk 2.3.0 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.43.0 which is incompatible.


In [ ]:
research_paper_filename = "research.pdf"

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

pdf = PyPDFLoader("research.pdf")
docs = pdf.load()

print(f"Loaded {len(docs)} pages")

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

chunks = splitter.split_documents(docs)
print(f"Created {len(chunks)} chunks")

embed = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"}
)

db = Chroma.from_documents(
    documents=chunks,
    embedding=embed,
    persist_directory="./chroma_db"
)

Loaded 5 pages
Created 40 chunks


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [ ]:
!pip install -q langchain-groq

import os
from langchain_groq import ChatGroq

os.environ["GROQ_API_KEY"] = "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"

model = ChatGroq(
    model_name="llama-3.1-8b-instant",
    temperature=0.1
)

In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

temp = """You are an assistant that answers questions about a research paper.

Use only the given context to answer the question. If the answer is not present, say "I don't see that information in the research paper."

Context:
{context}

Question:
{question}

Answer:"""

prompt = PromptTemplate(
    template=temp,
    input_variables=["context", "question"]
)

def get_text(docs):
    return "\n\n".join(doc.page_content for doc in docs)

retriever = db.as_retriever(search_kwargs={"k": 10})

chain = (
    {
        "context": retriever | get_text,
        "question": RunnablePassthrough()
    }
    | prompt
    | model
    | StrOutputParser()
)

In [ ]:
q = "What are the main applications of Artificial Intelligence discussed in this research paper?"

print("Question:", q)
print("\nSearching research paper...\n")

docs = retriever.invoke(q)
print(f"Found {len(docs)} relevant sections\n")

ans = chain.invoke(q)

print("Answer:\n")
print(ans)

print("\n" + "=" * 50)
print("Sources Used:\n")

for d in docs:
    text = d.page_content.replace("/envel⌢", "")
    print(text)
    print("-" * 50)

Question: What are the main applications of Artificial Intelligence discussed in this research paper?

Searching research paper...

Found 10 relevant sections

Answer:

The main applications of Artificial Intelligence discussed in this research paper are:

1. Healthcare (specifically in fields such as cardiology, neurology, and embryology)
2. Heavy Industries (for efficient and safe operation of huge machines)
3. Telecommunications (using heuristic search in the management of their operations)
4. Entertainment
5. Finance
6. Education

Sources Used:

References 
 
1. http://en.wikibooks.org/wiki/Computer_Science:Artificial_Intelligence 
http://www.howstuffworks.com/arificialintelligence 
2. http:// www.google.co.in 
3. http://www.library.thinkquest.org 
4. https://www.javatpoint.com/application-of-ai 
5. https://www.educba.com/artificial-intelligence-techniques/ 
6. https://www.cigionline.orgw/articles/cyber-security-
battlefield/?utm_source=google_ads&utm_medium=grant&gclid=EAIaIQobChM